# BLG454E Term Project - Fall 2024 - WhoGetThisIdea

- Orhan Berkay Yılmaz
  - yilmazor21@itu.edu.tr
  - 150210054

- Pınar Erçin
  - ercin21@itu.edu.tr
  - 150210336

- Uğur Erkan Bol
  - bol21@itu.edu.tr
  - 150210074

- Zeliha Melek Bekdemir
  - bekdemir22@itu.edu.tr
  - 150220031

# Import Data Set

In [1]:
from google.colab import drive
drive.mount('/content/drive')

# keeping a copy of data inside drive was easier
!cp /content/drive/MyDrive/blg-454-e-fall-2024-term-project/train_feats.npy /content/train_feats.npy
!cp /content/drive/MyDrive/blg-454-e-fall-2024-term-project/train_labels.csv /content/train_labels.csv
!cp /content/drive/MyDrive/blg-454-e-fall-2024-term-project/valtest_feats.npy /content/valtest_feats.npy
# !cp /content/drive/MyDrive/blg-454-e-fall-2024-term-project/sampleSubmission.csv /content/sampleSubmission.csv

Mounted at /content/drive


# Import libraries to be used

In [ ]:
import numpy as np
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC

from scipy.stats import mode
from scipy import stats
import random as r

# Load data

In [ ]:
#Loading data from the files

train_feats = np.load('./train_feats.npy', allow_pickle=True)
t = train_feats.item()
# idx = t['idx']

train_labels = np.loadtxt("./train_labels.csv",delimiter=",", dtype=str)
train_labels = train_labels[1:,1]
# l_idx = train_labels[1:,0]

#display(train_feats)
#display(train_labels)

valtest_feats = np.load('./valtest_feats.npy', allow_pickle=True)
v = valtest_feats.item()

# Final Model

Here is our final model using 5 fold cross validation. Other earlier model without cross validation is at one below.

## 5-Fold

In [ ]:
#Concatanating the data horizontally to get a matrix of data with features
train_data = np.concatenate(( t['resnet_feature'], t['vit_feature'], t['clip_feature'], t['dino_feature']), axis=1)
valtest_data = np.concatenate((v['resnet_feature'],v['vit_feature'],v['clip_feature'], v['dino_feature']), axis=1)

# Set random seed for reproducibility
np.random.seed(1)
random_state = round(np.random.random() * 1000)

# Initialize K-Fold cross-validation
kf = KFold(n_splits=5, shuffle=True, random_state=random_state)

# Store predictions and metrics
test_fold_accuracies = []
test_fold_f1_scores = []

# Initialize array to store all validation predictions
num_samples = len(valtest_data)
all_val_predictions = np.zeros((5, num_samples))

# Perform k-fold cross-validation
for i, (train_index, test_index) in enumerate(kf.split(train_data)):
    print(f"Fold {i + 1}:")

    # Split the data into training and testing sets for this fold
    X_train_fold = train_data[train_index]
    X_test_fold = train_data[test_index]
    y_train_fold = train_target[train_index]
    y_test_fold = train_target[test_index]

    # Train the model
    clf = make_pipeline(StandardScaler(), SVC(gamma='auto'))
    clf.fit(X_train_fold, y_train_fold)

    # Predict on test data for this fold
    y_test_pred = clf.predict(X_test_fold)

    # Calculate metrics for test data
    test_accuracy = accuracy_score(y_test_fold, y_test_pred)
    test_f1 = f1_score(y_test_fold, y_test_pred, average='weighted')

    test_fold_accuracies.append(test_accuracy)
    test_fold_f1_scores.append(test_f1)

    print(f"  Test accuracy: {test_accuracy:.4f}")
    print(f"  Test F1 score: {test_f1:.4f}")

    # Predict on validation data and store in the corresponding row
    val_pred_fold = clf.predict(valtest_data)
    all_val_predictions[i] = val_pred_fold

# Get majority vote for each validation sample
majority_predictions = stats.mode(all_val_predictions, axis=0)[0].flatten()

# Create final predictions array (20k x 2)
final_val_predictions = np.column_stack((np.arange(num_samples), majority_predictions))

# Calculate and display overall performance on test data
mean_test_accuracy = np.mean(test_fold_accuracies)
mean_test_f1 = np.mean(test_fold_f1_scores)

print(f"\nMean test accuracy across folds: {mean_test_accuracy:.4f}")
print(f"Mean test F1 score across folds: {mean_test_f1:.4f}")

# Print final predictions for validation data
print("\nFinal predicted labels for validation data (showing first 5 rows):")
print(final_val_predictions[:5])

# Print the distribution of predictions for first 5 validation samples
print("\nExample of predictions across folds for first 5 validation samples:")
for j in range(min(5, all_val_predictions.shape[1])):
    print(f"Sample {j}: Predictions across folds = {all_val_predictions[:, j]}, "
          f"Final prediction = {final_val_predictions[j, 1]}")

save_predictions_to_csv(final_val_predictions[:, 1], filename="output.csv")

## Last Model without Cross Validation

In [ ]:
#Concatanating the data horizontally to get a matrix of data with features
train_data = np.concatenate(( t['resnet_feature'], t['vit_feature'], t['clip_feature'], t['dino_feature']), axis=1)
valtest_data = np.concatenate((v['resnet_feature'],v['vit_feature'],v['clip_feature'], v['dino_feature']), axis=1)

X_train = train_data
y_train = train_labels

#SVM with Bagging
clf = make_pipeline(StandardScaler(), SVC(gamma='auto', C=2.5, kernel='rbf'))
clf = BaggingClassifier(estimator=clf, n_estimators=10, n_jobs=-1, random_state=0).fit(X_train, y_train)
y_pred = clf.predict(valtest_data)

def save_predictions_to_csv(y_pred, filename="output.csv"):
    output_df = pd.DataFrame({
        'ID': np.arange(len(y_pred)),  # Sequential IDs starting from 0
        'Predicted': y_pred.astype(int)  # Convert to integer type
    })
    output_df.to_csv(filename, index=False)

save_predictions_to_csv(y_pred)

# **After this line there are our earlier tries**


---

## Preprocessing

- Use one of the combinations below:
  - no processing
  - only normalize
  - only pca(seperate)
  - only pca(whole data)
  - normalize + pca(seperate)
  - normalize + pca(whole data)

- Our SVC pipeline already uses the scaler so no need to normalize earlier

In [ ]:
# Use all data without processing

train_data = np.concatenate(( t['resnet_feature'], t['vit_feature'], t['clip_feature'], t['dino_feature']), axis=1)
valtest_data = np.concatenate((v['resnet_feature'],v['vit_feature'],v['clip_feature'], v['dino_feature']), axis=1)

In [ ]:
def apply_norm_and_transform(scaler, feature, val_feature):
    feature_transformed = scaler.fit_transform(feature)
    val_feature_transformed = scaler.transform(val_feature)
    return feature_transformed, val_feature_transformed

In [ ]:
# Normalize data

scaler = StandardScaler()
norm_resnet, norm_val_resnet = apply_norm_and_transform(scaler, t['resnet_feature'], v['resnet_feature'])
norm_vit, norm_val_vit = apply_norm_and_transform(scaler, t['vit_feature'], v['vit_feature'])
norm_clip, norm_val_clip = apply_norm_and_transform(scaler, t['clip_feature'], v['clip_feature'])
norm_dino, norm_val_dino = apply_norm_and_transform(scaler, t['dino_feature'], v['dino_feature'])

train_data = np.concatenate((norm_resnet, norm_vit, norm_clip, norm_dino), axis=1)
valtest_data = np.concatenate((norm_val_resnet, norm_val_vit, norm_val_clip, norm_val_dino), axis=1)

In [ ]:
def apply_pca_and_transform(pca, feature, val_feature):
    feature_transformed = pca.fit_transform(feature)
    val_feature_transformed = pca.transform(val_feature)
    return feature_transformed, val_feature_transformed

In [ ]:
# PCA for each class seperately

pca = PCA(n_components=192)

# Apply PCA to all features
pca_resnet, pca_val_resnet = apply_pca_and_transform(pca, t['resnet_feature'], v['resnet_feature'])
pca_vit, pca_val_vit = apply_pca_and_transform(pca, t['vit_feature'], v['vit_feature'])
pca_clip, pca_val_clip = apply_pca_and_transform(pca, t['clip_feature'], v['clip_feature'])
pca_dino, pca_val_dino = apply_pca_and_transform(pca, t['dino_feature'], v['dino_feature'])

# Concatenate the features for training and validation/testing
train_data = np.concatenate((pca_resnet, pca_vit, pca_clip, pca_dino), axis=1)
valtest_data = np.concatenate((pca_val_resnet, pca_val_vit, pca_val_clip, pca_val_dino), axis=1)

In [ ]:
# PCA for whole data together

pca = PCA(n_components=768)

whole_data = np.concatenate((t['resnet_feature'], t['vit_feature'], t['clip_feature'], t['dino_feature']), axis=1)
whole_val_data = np.concatenate((v['resnet_feature'],v['vit_feature'],v['clip_feature'], v['dino_feature']), axis=1)

pca_whole_data, pca_whole_val_data = apply_pca_and_transform(pca, whole_data, whole_val_data)

train_data = pca_whole_data
valtest_data = pca_whole_val_data

In [ ]:
# Normalize and PCA for each group

# Normalize
scaler = StandardScaler()
norm_resnet, norm_val_resnet = apply_norm_and_transform(scaler, t['resnet_feature'], v['resnet_feature'])
norm_vit, norm_val_vit = apply_norm_and_transform(scaler, t['vit_feature'], v['vit_feature'])
norm_clip, norm_val_clip = apply_norm_and_transform(scaler, t['clip_feature'], v['clip_feature'])
norm_dino, norm_val_dino = apply_norm_and_transform(scaler, t['dino_feature'], v['dino_feature'])

# Apply PCA to all features
pca = PCA(n_components=192)
pca_norm_resnet, pca_norm_val_resnet = apply_pca_and_transform(pca, norm_resnet, norm_val_resnet)
pca_norm_vit, pca_norm_val_vit = apply_pca_and_transform(pca, norm_vit, norm_val_vit)
pca_norm_clip, pca_norm_val_clip = apply_pca_and_transform(pca, norm_clip, norm_val_clip)
pca_norm_dino, pca_norm_val_dino = apply_pca_and_transform(pca, norm_dino, norm_val_dino)

train_data = np.concatenate((pca_norm_resnet, pca_norm_vit, pca_norm_clip, pca_norm_dino), axis=1)
valtest_data = np.concatenate((pca_norm_val_resnet, pca_norm_val_vit, pca_norm_val_clip, pca_norm_val_dino), axis=1)

In [ ]:
# Normalize and PCA for whole data

# Normalize
scaler = StandardScaler()
norm_resnet, norm_val_resnet = apply_norm_and_transform(scaler, t['resnet_feature'], v['resnet_feature'])
norm_vit, norm_val_vit = apply_norm_and_transform(scaler, t['vit_feature'], v['vit_feature'])
norm_clip, norm_val_clip = apply_norm_and_transform(scaler, t['clip_feature'], v['clip_feature'])
norm_dino, norm_val_dino = apply_norm_and_transform(scaler, t['dino_feature'], v['dino_feature'])

# PCA
pca = PCA(n_components=768)

whole_data = np.concatenate((norm_resnet, norm_vit, norm_clip, norm_dino), axis=1)
whole_val_data = np.concatenate((norm_val_resnet, norm_val_vit, norm_val_clip, norm_val_dino), axis=1)

pca_whole_data, pca_whole_val_data = apply_pca_and_transform(pca, whole_data, whole_val_data)

train_data = pca_whole_data
valtest_data = pca_whole_val_data

In [ ]:
# Check the shape of the data
print(f"Train Data: {train_data.shape}")
print(f"Validation/Test Data: {valtest_data.shape}")

assert(train_data.shape[0] == train_labels.shape[0])
assert(valtest_data.shape[0] == 20000)

Train Data: (40000, 768)
Validation/Test Data: (20000, 768)


### To test or not to test

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(train_data, train_labels, test_size=0.1)

In [ ]:
X_train = train_data
y_train = train_labels

X_test = []
y_test = []

## Utils

In [ ]:
def save_predictions_to_csv(y_pred, filename="output.csv"):
    output_df = pd.DataFrame({
        'ID': np.arange(len(y_pred)),  # Sequential IDs starting from 0
        'Predicted': y_pred.astype(int)  # Convert to integer type
    })
    output_df.to_csv(filename, index=False)

# Pass only the second column of final_val_predictions to the function

## Tested Other Models

### Basic Model Tests

In [ ]:
# For finding a baseline GNB and KNN
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier

gnb = GaussianNB()
gnb.fit(X_train, y_train)
y_pred = gnb.predict(X_test)

print(classification_report(y_test, y_pred))

knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)
y_pred = knn.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.98      0.98      0.98       394
           1       0.99      0.98      0.99       381
           2       0.96      0.98      0.97       402
           3       0.94      0.96      0.95       418
           4       0.98      0.97      0.97       430
           5       0.97      0.97      0.97       413
           6       1.00      0.98      0.99       397
           7       0.99      0.98      0.99       390
           8       0.99      0.99      0.99       381
           9       0.98      0.99      0.98       394

    accuracy                           0.98      4000
   macro avg       0.98      0.98      0.98      4000
weighted avg       0.98      0.98      0.98      4000

0.977
              precision    recall  f1-score   support

           0       0.98      0.99      0.98       394
           1       0.99      0.99      0.99       381
           2       0.99      0.98      0.98       402
           3      

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

# was still running after 30 mins with n_estimators=100
clf = GradientBoostingClassifier(n_estimators=10, learning_rate=1.0, max_depth=1, random_state=0).fit(X_train, y_train)
clf.score(X_test, y_test)

0.33325

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

#pca 100x4 < pca 20x4 < pca 50x4 = pca 40x4
clf = HistGradientBoostingClassifier(random_state=0).fit(X_train, y_train)
clf.score(X_test, y_test)

0.98375

In [ ]:
from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier(random_state=0).fit(X_train, y_train)
clf.score(X_test, y_test)

0.98225

In [ ]:
from sklearn.tree import DecisionTreeClassifier

clf = DecisionTreeClassifier(random_state=0).fit(X_train, y_train)
clf.score(X_test, y_test)

0.95725

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.datasets import make_classification

# 18 mins and still counting with 1000, 100 was too bad
clf = AdaBoostClassifier(n_estimators=100, random_state=0).fit(X_train, y_train)
clf.score(X_test, y_test)

/usr/local/lib/python3.10/dist-packages/sklearn/ensemble/_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(


0.651

In [ ]:
from sklearn.linear_model import LogisticRegression

clf = LogisticRegression(random_state=0, max_iter=1000).fit(X_train, y_train)
clf.score(X_test, y_test)

0.98375

### MLP Test

In [ ]:
  #An mlp try with pca

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import KFold
import matplotlib.pyplot as plt


def load_data():
    # Load features
    train_feats = np.load("C:\\Users\\zmbek\\Downloads\\train_feats.npy", allow_pickle=True).item()
    resnet_features = train_feats['resnet_feature']
    vit_features = train_feats['vit_feature']
    clip_features = train_feats['clip_feature']
    dino_features = train_feats['dino_feature']

    all_features = np.hstack((resnet_features, vit_features, clip_features, dino_features)).astype(np.float32)
    np.save("C:\\Users\\zmbek\\Downloads\\all_features.npy", all_features)

    # Load labels
    labels = pd.read_csv("C:\\Users\\zmbek\\Downloads\\train_labels.csv")
    label_encoder = LabelEncoder()
    y = label_encoder.fit_transform(labels['label'])  # Assuming 'label' column
    return all_features, y

def pca_with_svd(data, n_components=None):
    data_tensor = torch.tensor(data, dtype=torch.float32).to("cuda")
    data_mean = torch.mean(data_tensor, dim=0)
    centered_data = data_tensor - data_mean

    # SVD
    U, S, Vt = torch.linalg.svd(centered_data, full_matrices=False)
    if n_components:
        Vt = Vt[:n_components, :]

    # Project data
    projected_data = torch.mm(centered_data, Vt.T).cpu().numpy()

    # Return PCA components and mean
    return projected_data, data_mean.cpu().numpy(), Vt.cpu().numpy()

def predict_and_save_final(model, validation_features_path, output_csv_path, device="cuda"):
    # Load validation features
    val_feats = np.load(validation_features_path, allow_pickle=True).item()
    resnet_features = val_feats['resnet_feature']
    vit_features = val_feats['vit_feature']
    clip_features = val_feats['clip_feature']
    dino_features = val_feats['dino_feature']
    validation_features = np.hstack((resnet_features, vit_features, clip_features, dino_features)).astype(np.float32)

    # Load PCA components and mean
    pca_mean = np.load("C:\\Users\\zmbek\\Downloads\\pca_mean.npy")
    pca_components = np.load("C:\\Users\\zmbek\\Downloads\\pca_components.npy")

    # Apply PCA to validation features
    validation_features_centered = validation_features - pca_mean
    projected_validation_features = np.dot(validation_features_centered, pca_components.T)

    # Convert to PyTorch tensor
    val_tensor = torch.tensor(projected_validation_features, dtype=torch.float32).to(device)

    # Perform inference
    model.eval()
    with torch.no_grad():
        predictions = model(val_tensor)
        _, predicted_labels = torch.max(predictions, 1)

    # Create DataFrame with ID and Predicted columns
    predicted_labels = predicted_labels.cpu().numpy()
    output_df = pd.DataFrame({
        'ID': np.arange(len(predicted_labels)),  # Assuming sequential IDs starting from 0
        'Predicted': predicted_labels
    })

    # Save to CSV
    output_df.to_csv(output_csv_path, index=False)

    return output_df
class MLPModel(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(MLPModel, self).__init__()
        self.sequence = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, output_dim)
        )

    def forward(self, x):
        return self.sequence(x)

def train_fold(model, train_loader, val_loader, optimizer, criterion, epochs=10, device="cuda"):
    model = model.to(device)
    train_errors = []  # To store training errors for each epoch
    val_errors = []    # To store validation errors for each epoch

    for epoch in range(epochs):
        model.train()
        correct_train = 0
        total_train = 0
        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(features)
            _, predicted = torch.max(outputs, 1)
            total_train += labels.size(0)
            correct_train += (predicted == labels).sum().item()

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

        train_accuracy = correct_train / total_train
        train_error = 1 - train_accuracy
        train_errors.append(train_error)
        print(f"Epoch {epoch + 1}/{epochs}, Train Error: {train_error:.4f}")

        # Validate the model
        model.eval()
        correct_val = 0
        total_val = 0
        with torch.no_grad():
            for features, labels in val_loader:
                features, labels = features.to(device), labels.to(device)
                outputs = model(features)
                _, predicted = torch.max(outputs, 1)
                total_val += labels.size(0)
                correct_val += (predicted == labels).sum().item()

        val_accuracy = correct_val / total_val
        val_error = 1 - val_accuracy
        val_errors.append(val_error)
        print(f"Validation Error: {val_error:.4f}")

    return train_errors, val_errors


def train_foldx(model, train_loader, val_loader, optimizer, criterion, epochs=10, device="cuda"):
    model = model.to(device)
    train_losses = []
    val_accuracies = []

    for epoch in range(epochs):
        # Training loop
        model.train()
        train_loss = 0.0
        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)
        train_losses.append(train_loss)

        # Validation loop
        model.eval()
        correct = 0
        total = 0
        with torch.no_grad():
            for features, labels in val_loader:
                features, labels = features.to(device), labels.to(device)
                outputs = model(features)
                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        val_accuracy = correct / total
        val_accuracies.append(val_accuracy)

        print(f"Epoch {epoch + 1}/{epochs}, Train Loss: {train_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}")

    return train_losses, val_accuracies

from sklearn.metrics import f1_score

def train_foldy(model, train_loader, val_loader, optimizer, criterion, epochs=10, device="cuda"):
    model = model.to(device)
    train_losses = []
    val_accuracies = []
    val_f1_scores = []  # To store F1 scores for validation sets

    for epoch in range(epochs):
        # Training loop
        model.train()
        train_loss = 0.0
        for features, labels in train_loader:
            features, labels = features.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(features)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)
        train_losses.append(train_loss)

        # Validation loop
        model.eval()
        all_preds = []
        all_labels = []
        with torch.no_grad():
            for features, labels in val_loader:
                features, labels = features.to(device), labels.to(device)
                outputs = model(features)
                _, predicted = torch.max(outputs, 1)
                all_preds.extend(predicted.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        # Calculate accuracy and F1 score
        val_accuracy = np.mean(np.array(all_preds) == np.array(all_labels))
        val_accuracies.append(val_accuracy)
        val_f1 = f1_score(all_labels, all_preds, average='weighted')  # Weighted F1 score
        val_f1_scores.append(val_f1)

        print(f"Epoch {epoch + 1}/{epochs}, Train Loss: {train_loss:.4f}, Validation Accuracy: {val_accuracy:.4f}, F1 Score: {val_f1:.4f}")

    return train_losses, val_accuracies, val_f1_scores


def run():
    # Load and preprocess data
    all_features, y = load_data()

    n_components = 768
    projected_features, pca_mean, pca_components = pca_with_svd(all_features, n_components=n_components)

        # Save PCA components for future use
    np.save("C:\\Users\\zmbek\\Downloads\\pca_mean.npy", pca_mean)
    np.save("C:\\Users\\zmbek\\Downloads\\pca_components.npy", pca_components)

    # Save projected features
    np.save("C:\\Users\\zmbek\\Downloads\\projected_features.npy", projected_features)

    # Convert to PyTorch tensors
    X = torch.tensor(projected_features, dtype=torch.float32)
    y = torch.tensor(y, dtype=torch.long)

    # 5-Fold Cross-Validation
    kfold = KFold(n_splits=5, shuffle=True, random_state=42)
    fold_train_errors = []
    fold_val_errors = []
    fold_f1_scores = []  # To store F1 scores per fold

    for fold, (train_idx, val_idx) in enumerate(kfold.split(X)):
        print(f"Fold {fold + 1}")

        # Create DataLoader for training and validation
        train_dataset = TensorDataset(X[train_idx], y[train_idx])
        val_dataset = TensorDataset(X[val_idx], y[val_idx])
        train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

        # Initialize model, optimizer, and loss function
        input_dim = projected_features.shape[1]
        output_dim = len(torch.unique(y))
        model = MLPModel(input_dim, output_dim)
        optimizer = optim.Adam(model.parameters(), lr=0.001)
        criterion = nn.CrossEntropyLoss()

        # Train and evaluate the model
        train_losses, val_accuracies, val_f1_scores = train_foldy(model, train_loader, val_loader, optimizer, criterion, epochs=10)

        # Record metrics for the fold
        avg_train_error = train_losses[-1]
        avg_val_error = 1 - val_accuracies[-1]
        avg_f1_score = val_f1_scores[-1]

        fold_train_errors.append(avg_train_error)
        fold_val_errors.append(avg_val_error)
        fold_f1_scores.append(avg_f1_score)

        print(f"Fold {fold + 1}, Final F1 Score: {avg_f1_score:.4f}")

    # Plot F1 scores across folds
    plt.figure(figsize=(8, 6))
    plt.bar(range(1, 6), fold_f1_scores, color='skyblue')
    plt.title('F1 Score per Fold')
    plt.xlabel('Fold')
    plt.ylabel('F1 Score')
    plt.xticks(range(1, 6))
    plt.tight_layout()
    plt.show()

    # Predict and save
    validation_features_path = "C:\\Users\\zmbek\\Downloads\\valtest_feats.npy"
    output_csv_path = "C:\\Users\\zmbek\\Downloads\\predicted_labels.csv"
    predict_and_save_final(model, validation_features_path, output_csv_path)

run()

### SVM

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

clf = make_pipeline(StandardScaler(), SVC(gamma='auto', C=2.5, kernel='rbf')).fit(X_train, y_train)
clf.score(X_test, y_test)

0.99075

### SVM with Bagging

In [ ]:
from sklearn.ensemble import BaggingClassifier

clf = make_pipeline(StandardScaler(), SVC())
clf = BaggingClassifier(estimator=clf, n_estimators=10, n_jobs=-1, random_state=0).fit(X_train, y_train) #0.9867
clf.score(X_test, y_test)

0.98825

## Hyperparameter Tests

### HGBC hyperparameter

In [ ]:
# PCA hyperparameter

for i in range(20):
  pca = PCA(n_components=(i+1)*10)
  pca_resnet_feature = pca.fit_transform(resnet_feature)
  #v_resnet_feature = pca.transform(v['resnet_feature'])

  pca_vit_feature = pca.fit_transform(vit_feature)
  #v_vit_feature = pca.transform(v['vit_feature'])

  pca_clip_feature = pca.fit_transform(clip_feature)
  #v_clip_feature = pca.transform(v['clip_feature'])

  pca_dino_feature = pca.fit_transform(dino_feature)
  #v_dino_feature = pca.transform(v['dino_feature'])

  train_data = np.concatenate((pca_resnet_feature, pca_vit_feature, pca_clip_feature, pca_dino_feature), axis=1)
  #valtest_data = np.concatenate((v_resnet_feature, v_vit_feature, v_clip_feature, v_dino_feature), axis=1)

  X_train, X_test, y_train, y_test = train_test_split(train_data, train_target, test_size=0.2)

  clf = HistGradientBoostingClassifier(random_state=0).fit(X_train, y_train)

  print((i+1)*5, ": ", clf.score(X_test, y_test))

0 :  0.960875
1 :  0.98025
2 :  0.981375
3 :  0.9835
4 :  0.98275
5 :  0.98325
6 :  0.985125
7 :  0.98325
8 :  0.98375
9 :  0.98225
10 :  0.9865
11 :  0.984
12 :  0.9845
13 :  0.983


KeyboardInterrupt: 

In [ ]:
!pip install optuna
import optuna

def objective(trial):
    x = trial.suggest_int('x', 5, 100)
    pca = PCA(n_components=x)

    pca_resnet_feature = pca.fit_transform(resnet_feature)
    pca_vit_feature = pca.fit_transform(vit_feature)
    pca_clip_feature = pca.fit_transform(clip_feature)
    pca_dino_feature = pca.fit_transform(dino_feature)

    X_train, X_test, y_train, y_test = train_test_split(train_data, train_target, test_size=0.2)

    clf = HistGradientBoostingClassifier(random_state=0).fit(X_train, y_train)

    return clf.score(X_test, y_test)

study = optuna.create_study()
study.optimize(objective, n_trials=100)

study.best_params

[I 2024-12-11 17:40:00,794] A new study created in memory with name: no-name-bdba1951-13f4-41e2-8707-79eb1fcb8c74
[I 2024-12-11 17:41:14,260] Trial 0 finished with value: 0.987125 and parameters: {'x': 44}. Best is trial 0 with value: 0.987125.
[I 2024-12-11 17:42:16,385] Trial 1 finished with value: 0.983375 and parameters: {'x': 12}. Best is trial 1 with value: 0.983375.
[I 2024-12-11 17:43:13,458] Trial 2 finished with value: 0.98375 and parameters: {'x': 42}. Best is trial 1 with value: 0.983375.
[I 2024-12-11 17:44:18,785] Trial 3 finished with value: 0.984 and parameters: {'x': 20}. Best is trial 1 with value: 0.983375.
[I 2024-12-11 17:45:25,392] Trial 4 finished with value: 0.98425 and parameters: {'x': 80}. Best is trial 1 with value: 0.983375.
[I 2024-12-11 17:46:33,358] Trial 5 finished with value: 0.9855 and parameters: {'x': 61}. Best is trial 1 with value: 0.983375.
[I 2024-12-11 17:47:33,616] Trial 6 finished with value: 0.984875 and parameters: {'x': 80}. Best is trial 

KeyboardInterrupt: 

### SVM Classifier Hyperparameter Optimization

In [ ]:
!pip install optuna
import optuna

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

def objective(trial):
    c = trial.suggest_float('C', 0.01, 10000)
    d = trial.suggest_int('degree', 2, 10)
    g = trial.suggest_categorical('gamma', ['scale', 'auto'])
    k = trial.suggest_categorical('kernel', ['linear', 'poly', 'rbf']) # sigmoid dümdüz kötü # rbf default, poly 2. aday

    clf = make_pipeline(StandardScaler(), SVC(C=c, degree=d, kernel=k, gamma=g)).fit(X_train, y_train)
    return (1-clf.score(X_test, y_test))*100

study = optuna.create_study()
study.optimize(objective, n_trials=50, n_jobs=6)

study.best_params